# NeurIPS 2026 Smart Buildings — Submission Version 10

**Independent experiment:** critical-terminal **trim-and-respond** duct-pressure reset. Rather than computing fan pressure directly from the current zone error (V2, V5, V7–V9) or learning a thermal model (V6), V10 deliberately trims an AHU's pressure on successive 5-minute steps until its VAV terminals report insufficient air delivery. One or more starved terminals then request an asymmetric pressure increase. The idea is related to Guideline 36's trim-and-respond sequence, adapted conservatively for this simulator and **not** a claim of conformance.

## Component responsibilities

| Method | Responsibility |
| --- | --- |
| `__init__`, `_topology`, `_find_field` | Cache per-building action indices and optional AHU airflow sensors |
| `_fallback_band`, `_zone_error` | Validate observed temperatures and live comfort setpoints |
| `_terminal_command` | Command zone dampers/reheat; report high-demand saturated terminals |
| `_pressure_reset` | Slowly trim fan setpoint; rapidly respond to saturation or flow deficit |
| `_control_ahu` | AHU run and cooling-only supply setpoint, with independent pressure-reset loop |
| `_control_water` | Activate shared heating loop for VAV reheat needs |
| `_idle`, `_write` | Enforce hybrid action normalization and true idle commands |
| `reset`, `get_action` | Clear per-episode states; do O(AHUs + zones) runtime dispatch |

**Key distinctions:** no PID or integral error accumulation, no temperature forecasting, no occupancy-weighted run thresholds, no RLS identification/candidate search, no outdoor-air economizer, and no on/off dwell timer. V10 carries only a pressure command per AHU, not an estimated zone thermal state. It uses optional matched AHU airflow (sensed and setpoint) only if both are present; otherwise the commanded VAV saturation signal alone drives reset.

**Limitations:** this is an exploratory controller, not a calibrated Guideline 36 sequence. Saturated commanded dampers approximate starved delivery, not measured terminal air flow. Fan-pressure and ventilation floors still apply. The synthetic contract tests do not measure comfort/cost/carbon or official score. The lowest recorded public score in submission history remains V7 at 0.03947 (lower is better).

**Run checks:** `python Co_Kaggle/g8/submissions/test_version10.py`. Full simulator and Kaggle evaluation are required before claiming improvement.


In [ ]:
"""Version 10: critical-terminal trim-and-respond duct pressure reset.

Unlike earlier controllers, each AHU learns no thermal model and runs no PID:
it gradually reduces fan pressure until a terminal requests more delivery, then
responds quickly. A high damper command represents unresolved air demand;
optional measured AHU airflow shortfall reinforces that request. The reset
state is isolated per AHU and cleared between episodes.
"""
import re

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Topology-independent HVAC control with terminal-request pressure reset."""

    START_PRESSURE_PA = 700.0
    MIN_PRESSURE_PA = 200.0
    MAX_PRESSURE_PA = 1600.0
    TRIM_PRESSURE_PA = 25.0
    RESPOND_PRESSURE_PA = 120.0
    CRITICAL_DAMPER = 0.88
    CRITICAL_ERROR_K = 0.60
    RUN_ERROR_K = 0.25
    SEVERE_ERROR_K = 4.0

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        meta = building_meta_data
        self.n_continuous = int(meta["n_continuous_action"])
        self.n_discrete = int(meta["n_discrete_action"])
        self.fields = {str(name): i for i, name in
                       enumerate(meta["observation_field_names"])}
        self.supply = list(map(int, meta["temp_indices"]))
        self.fans = list(map(int, meta["pressure_indices"]))
        self.returns = list(map(int, meta["recirculation_indices"]))
        self.waters = list(map(int, meta["water_indices"]))
        self.n_ahu = len(self.supply)
        self.water_run = self.n_discrete - 1
        self.water_supply = self.waters[0] if self.waters else None
        self.water_pressure = (self.fans[self.n_ahu]
                               if len(self.fans) > self.n_ahu else None)
        self.comfort = tuple(map(float, meta["comfort_temp_window"]))
        self.eco = tuple(map(float, meta["eco_temp_window"]))
        normalization = meta.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {})
        self.zone_mean = float(normalization.get("sample_mean", 295.0))
        self.zone_std = float(normalization.get("sample_std", 1.0))
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0
        self.ahus = self._topology(meta)
        self.reset()

    @staticmethod
    def _numeric_order(value):
        match = re.search(r"\d+", str(value))
        return (int(match.group()) if match else -1, str(value))

    def _topology(self, meta):
        """Map each AHU and its VAV sensors to actuators once at startup."""
        ahus = []
        for ahu_id, ahu_name in enumerate(sorted(meta["ahu_zone_assignments"],
                                                 key=self._numeric_order)):
            if ahu_id >= self.n_ahu:
                break
            zones = []
            for room in meta["ahu_zone_assignments"][ahu_name]:
                name = str(room).replace("room_", "VAV_")
                temp = self.fields.get(name + "_zone_air_temperature_sensor")
                if temp is None:
                    continue
                indices = meta["zone_action_indices"].get(name, {})
                zones.append({
                    "temp": temp,
                    "heat": self.fields.get(name + "_zone_air_heating_temperature_setpoint_sensor"),
                    "cool": self.fields.get(name + "_zone_air_cooling_temperature_setpoint_sensor"),
                    "damper": indices.get("supply_air_damper_percentage_command"),
                    "reheat": indices.get("reheat_valve_percentage_command"),
                })
            ahu_name = str(ahu_name)
            prefix = "ahs_" + ahu_name + "_"
            ahus.append({
                "run": ahu_id,
                "supply": self.supply[ahu_id],
                "fan": self.fans[ahu_id] if ahu_id < len(self.fans) else None,
                "return": self.returns[ahu_id] if ahu_id < len(self.returns) else None,
                "airflow": self._find_field(prefix, ("supply_air_flowrate_sensor",
                                                       "supply_air_flow_rate_sensor")),
                "flow_target": self._find_field(prefix, ("supply_air_flowrate_setpoint",
                                                           "supply_air_flow_rate_setpoint")),
                "zones": zones,
            })
        return ahus

    def _find_field(self, prefix, endings):
        """Optional instrument: missing telemetry must not disable control."""
        for ending in endings:
            index = self.fields.get(prefix + ending)
            if index is not None:
                return index
        return None

    @staticmethod
    def _clamp(value, minimum, maximum):
        return float(np.clip(value, minimum, maximum))

    @classmethod
    def _fraction(cls, value):
        return 2.0 * cls._clamp(value, 0.0, 1.0) - 1.0

    @classmethod
    def _supply_temperature(cls, kelvin):
        return cls._clamp((kelvin - 295.0) / 10.0, -1.0, 1.0)

    def _read(self, observation, index):
        if index is None or not 0 <= index < len(observation):
            return float("nan")
        return float(observation[index])

    def _fallback_band(self, observation):
        """Use current comfort flag only; do not infer per-zone occupancy."""
        flag = self._read(observation, self.fields.get("comfort_mode_now"))
        return self.comfort if np.isfinite(flag) and flag >= 0.5 else self.eco

    def _zone_error(self, observation, zone, fallback):
        """Convert standardized temperature to Kelvin and validate setpoints."""
        raw = self._read(observation, zone["temp"])
        temperature = self.zone_mean + self.zone_std * raw
        if not np.isfinite(temperature) or not 240.0 <= temperature <= 340.0:
            return None
        low = self._read(observation, zone["heat"])
        high = self._read(observation, zone["cool"])
        if not (np.isfinite(low) and np.isfinite(high)
                and 245.0 <= low < high <= 335.0 and high - low <= 35.0):
            low, high = fallback
        return {"action": zone,
                "heating": max(0.0, low - temperature),
                "cooling": max(0.0, temperature - high)}

    def _idle(self):
        """Return truly idle actions, not normalized midpoint commands."""
        cont = np.full(self.n_continuous, -1.0, dtype=np.float32)
        disc = np.zeros(self.n_discrete, dtype=np.int32)
        for index in self.supply:
            self._write(cont, index, 1.0)
        for index in self.returns:
            self._write(cont, index, 1.0)
        return cont, disc

    def _write(self, action, index, value):
        if index is not None and 0 <= index < len(action):
            action[index] = self._clamp(value, -1.0, 1.0)

    def _terminal_command(self, zone, action, water):
        """Assign zone damper/reheat and identify air-starved terminals."""
        heating, cooling = zone["heating"], zone["cooling"]
        damper = 0.10
        valve = 0.0
        if cooling >= self.RUN_ERROR_K:
            damper = self._clamp(0.17 + 0.27 * cooling, 0.17, 1.0)
        elif heating >= self.RUN_ERROR_K:
            damper = self._clamp(0.28 + 0.20 * heating, 0.28, 1.0)
            if zone["action"]["reheat"] is not None:
                valve = self._clamp(0.18 + 0.28 * heating, 0.18, 1.0)
                water.append((heating, valve))
        self._write(action, zone["action"]["damper"], self._fraction(damper))
        self._write(action, zone["action"]["reheat"], self._fraction(valve))
        needs_air = max(heating, cooling) >= self.CRITICAL_ERROR_K
        return needs_air and damper >= self.CRITICAL_DAMPER

    def _pressure_reset(self, ahu, observation, critical_requests, severe):
        """Trim pressure unless zones ask for more delivery; respond quickly.

        Only compare AHU flow sensor against its OWN setpoint, if both exist and
        contain finite positive values. Never combine normalized and physical
        flow fields from different instruments.
        """
        index = ahu["run"]
        sensed = self._read(observation, ahu["airflow"])
        target = self._read(observation, ahu["flow_target"])
        deficit = (np.isfinite(sensed) and np.isfinite(target)
                   and target > 0.0 and 0.0 <= sensed < 0.8 * target)
        requests = int(critical_requests) + int(bool(deficit))
        if severe:
            requests = max(1, requests)
        delta = (self.RESPOND_PRESSURE_PA * min(requests, 2)
                 if requests else -self.TRIM_PRESSURE_PA)
        self.pressure_pa[index] = self._clamp(
            self.pressure_pa[index] + delta,
            self.MIN_PRESSURE_PA, self.MAX_PRESSURE_PA)
        return self.pressure_pa[index]

    def _control_ahu(self, ahu, observation, statuses, action, discrete, water):
        """Run thermal plant only for violating zones; reset fan independently."""
        if not statuses:
            self.pressure_pa[ahu["run"]] = self.START_PRESSURE_PA
            return
        worst = max(max(z["heating"], z["cooling"]) for z in statuses)
        if worst < self.RUN_ERROR_K:
            self.pressure_pa[ahu["run"]] = self.START_PRESSURE_PA
            return

        discrete[ahu["run"]] = 1
        hot = max(z["cooling"] for z in statuses)
        # AHU coil cools only; heating is supplied by VAV hot-water reheat.
        supply_k = self._clamp(298.0 - 1.8 * hot, 285.0, 305.0) if hot else 305.0
        self._write(action, ahu["supply"], self._supply_temperature(supply_k))
        self._write(action, ahu["return"], 1.0)
        critical = sum(self._terminal_command(z, action, water) for z in statuses)
        pressure_pa = self._pressure_reset(
            ahu, observation, critical, worst >= self.SEVERE_ERROR_K)
        self._write(action, ahu["fan"], pressure_pa / 1000.0 - 1.0)

    def _control_water(self, water, action, discrete):
        """Match shared reheat-loop availability to requested zone heating."""
        if not water or not 0 <= self.water_run < self.n_discrete:
            return
        discrete[self.water_run] = 1
        worst = max(error for error, _ in water)
        mean_valve = sum(valve for _, valve in water) / len(water)
        temp_k = self._clamp(323.0 + 4.0 * worst, 320.0, 348.0)
        self._write(action, self.water_supply, self._fraction((temp_k - 310.0) / 40.0))
        self._write(action, self.water_pressure,
                    self._fraction(self._clamp(0.40 + 0.50 * mean_valve,
                                               0.40, 0.95)))

    def reset(self):
        """The evaluator may reuse one policy between multiple episodes."""
        self.pressure_pa = [self.START_PRESSURE_PA] * self.n_ahu

    def get_action(self, obs):
        """One linear scan of the current building, with O(AHU) memory."""
        observation = np.asarray(obs, dtype=np.float64).reshape(-1)
        action, discrete = self._idle()
        fallback = self._fallback_band(observation)
        water = []
        for ahu in self.ahus:
            zones = [self._zone_error(observation, z, fallback)
                     for z in ahu["zones"]]
            statuses = [z for z in zones if z is not None]
            self._control_ahu(ahu, observation, statuses, action, discrete, water)
        self._control_water(water, action, discrete)
        np.nan_to_num(action, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(action, -1.0, 1.0, out=action)
        return {"continuous_action": action, "discrete_action": discrete}


In [ ]:
# Official Kaggle notebook inference gateway.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
